[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/81_async_timeout_gather_solution.ipynb)

# 🟡 Solution: Async Gather with Timeout Defaults

Reference solution for `async_timeout_gather`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import asyncio


In [ ]:
# ✅ SOLUTION

async def gather_with_timeout(coros, timeout: float, default=None):
    async def run_one(coro):
        try:
            return await asyncio.wait_for(coro, timeout=timeout)
        except asyncio.TimeoutError:
            return default
    return await asyncio.gather(*(run_one(coro) for coro in coros))


In [ ]:
# Verify

def run_async(coro):
    import asyncio
    import threading
    box = {}
    def target():
        try:
            box['value'] = asyncio.run(coro)
        except BaseException as e:
            box['error'] = e
    t = threading.Thread(target=target)
    t.start()
    t.join(timeout=5)
    if t.is_alive():
        raise TimeoutError('async test timed out')
    if 'error' in box:
        raise box['error']
    return box.get('value')

async def work(delay, value):
    await asyncio.sleep(delay)
    return value
print(run_async(gather_with_timeout([work(0.01, 1), work(0.1, 2)], timeout=0.03, default=None)))


In [ ]:
# Run judge
from torch_judge import check
check('async_timeout_gather')
